# Phase 5: Parametric Studies and Publication-Quality Results

This notebook performs a multi-dimensional CCE vs dose-rate sweep across epitaxial layer thickness, bulk doping concentration, and bias voltage for the FLASH SiC paper.

**Parameter space:** 4 epi thicknesses x 5 doping levels x 3 bias voltages = 60 conditions, each swept over 6 dose rates (20-230 Gy/s).

**Scientific goal:** Determine whether any combination of detector design parameters leads to measurable CCE degradation under FLASH dose rates in 4H-SiC.

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for notebook creation
import matplotlib.pyplot as plt
from src.flash_recombination import (parametric_cce_sweep, save_parametric_results,
                                      load_parametric_results, cce_vs_dose_rate)
from src.plotting import (plot_parametric_epi, plot_parametric_doping, plot_parametric_bias,
                          plot_cce_vs_dose_rate, save_figure)

print('All imports successful')

Searching DEVSIM_MATH_LIBS="libopenblas.dylib:liblapack.dylib:libblas.dylib"


Loading "libopenblas.dylib": MISSING DLL


Loading "liblapack.dylib": ALL BLAS/LAPACK LOADED


Skipping libblas.dylib


loading UMFPACK 5.1 as direct solver
All imports successful


## Configuration

Modify these parameters to customize the parametric sweep. Set `RECOMPUTE = True` to run the full sweep (expect ~1-2 hours for 48 conditions). With `RECOMPUTE = False`, previously cached results are loaded from disk.

In [2]:
DOSE_RATES = np.array([20, 50, 100, 150, 200, 230], dtype=float)
EPI_THICKNESSES = [5e-4, 10e-4, 15e-4, 20e-4]
N_D_BULK_VALUES = [5e13, 8.5e13, 1e14, 2e14, 5e14]
BIAS_VOLTAGES = [-10.0, -30.0, -50.0]
E_MEV = 62
RECOMPUTE = False
RESULTS_FILE = "../figures/parametric_results.json"

## Parametric Sweep

The sweep explores all combinations of the parameters above:
- **Epi thickness:** 5, 10, 15, 20 um
- **Bulk doping:** 5e13, 8.5e13, 1e14, 2e14, 5e14 cm$^{-3}$
- **Bias voltage:** -10, -30, -50 V

Total: 4 x 5 x 3 = 60 parameter combinations, each swept over 6 dose rates.

In [3]:
if RECOMPUTE:
    results = parametric_cce_sweep(DOSE_RATES, EPI_THICKNESSES, N_D_BULK_VALUES, BIAS_VOLTAGES, E_MEV)
    save_parametric_results(results, RESULTS_FILE)
    print(f"Sweep complete: {len(results)} conditions saved to {RESULTS_FILE}")
else:
    results = load_parametric_results(RESULTS_FILE)
    print(f"Loaded {len(results)} conditions from cache")

Loaded 1 conditions from cache


## Results: Epitaxial Thickness Dependence

CCE vs dose rate for varying epitaxial layer thickness at fixed reference doping ($N_D$ = 8.50e13 cm$^{-3}$). One panel per bias voltage.

In [4]:
fig = plot_parametric_epi(results, EPI_THICKNESSES, N_D_bulk_ref=8.50e13)
save_figure(fig, "flash_parametric_epi")
plt.show()

/Users/ngcex/projects/physics/petringa/notebooks/../src/plotting.py:789: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(title="Epi thickness", fontsize=8)


/var/folders/4v/3fndykhd0vq9b0wz8z3g72j80000gn/T/ipykernel_11623/4233188872.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Results: Doping Concentration Dependence

CCE vs dose rate for varying bulk doping concentration at fixed reference epi thickness (10 um). One panel per bias voltage.

In [5]:
fig = plot_parametric_doping(results, N_D_BULK_VALUES, epi_ref_cm=10e-4)
save_figure(fig, "flash_parametric_doping")
plt.show()

/Users/ngcex/projects/physics/petringa/notebooks/../src/plotting.py:852: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(title="Bulk doping", fontsize=8)


/var/folders/4v/3fndykhd0vq9b0wz8z3g72j80000gn/T/ipykernel_11623/4227558735.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Results: Bias Voltage Dependence

CCE vs dose rate for varying bias voltage at fixed reference epi thickness (10 um) and doping ($N_D$ = 8.50e13 cm$^{-3}$).

In [6]:
fig = plot_parametric_bias(results, BIAS_VOLTAGES, epi_ref_cm=10e-4, N_D_bulk_ref=8.50e13)
save_figure(fig, "flash_parametric_bias")
plt.show()

/var/folders/4v/3fndykhd0vq9b0wz8z3g72j80000gn/T/ipykernel_11623/3821699470.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary and Key Findings

This parametric study explores how detector design parameters affect charge collection under FLASH dose rates:

1. **Number of parameter combinations:** 60 (4 epi x 5 doping x 3 bias), each swept over 6 dose rates from 20 to 230 Gy/s.

2. **CCE degradation:** The analysis below checks whether any condition produces CCE below 0.99, indicating meaningful Auger-driven degradation.

3. **Implications for SiC dosimeters under FLASH irradiation:** If CCE remains near unity across all conditions, 4H-SiC detectors are robust dose-rate-independent dosimeters regardless of design parameter choice within the explored range.

---
*Phase 5 of the Petringa 4H-SiC TCAD simulation project*

In [7]:
total = len(results)
failed = sum(1 for v in results.values() if v is None)
print(f"Parameter combinations: {total}")
print(f"Failed simulations: {failed}")
print(f"Successful: {total - failed}")
# Check if any CCE < 0.99 (indicating meaningful degradation)
for key, val in results.items():
    if val is not None and np.any(np.array(val["cce_values"]) < 0.99):
        epi, nd, vb = key
        print(f"  CCE degradation at epi={epi*1e4:.0f}um, N_D={nd:.1e}, V={vb}V: min CCE={min(val['cce_values']):.4f}")

Parameter combinations: 1
Failed simulations: 0
Successful: 1
